In [ ]:
import pandas as pd

# Load the raw dataset
df = pd.read_csv("../data/raw/nyc311_raw.csv", low_memory=False)

# Basic shape check
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Check null percentage per column — helps confirm which columns are mostly empty
null_pct = df.isnull().mean().sort_values(ascending=False) * 100
print(null_pct.round(1))

In [ ]:
print("Number of unique complaint types:", df['complaint_type'].nunique())
print("\nTop 40 complaint types by count:")
print(df['complaint_type'].value_counts().head(40))

In [ ]:
# Module mapping based on agency + complaint_type keywords
module_map = {
    "Sanitation": {
        "agency": ["DSNY"],
        "keywords": ["dirty", "sanitation", "garbage", "recycling", "missed collection", "litter"]
    },
    "Water": {
        "agency": ["DEP"],
        "keywords": ["water", "sewer", "leak", "hydrant"]
    },
    "Electricity": {
        "agency": ["DOT", "CON EDISON"],
        "keywords": ["street light", "lamp", "electric", "power"]
    },
    "Roads": {
        "agency": ["DOT"],
        "keywords": ["street condition", "pothole", "sidewalk", "road", "highway", "curb"]
    },
    "Parks": {
        "agency": ["DPR"],
        "keywords": ["park", "tree", "playground"]
    },
    "Environment": {
        "agency": ["DEP", "DOHMH"],
        "keywords": ["air quality", "noise", "asbestos", "pollution", "hazardous"]
    }
}

def assign_module(row):
    text = str(row['complaint_type']).lower()
    agency = str(row['agency']).upper()
    for module, rule in module_map.items():
        if agency in rule["agency"] and any(k in text for k in rule["keywords"]):
            return module
    return None  # doesn't belong to any of our 6 modules

df['module'] = df.apply(assign_module, axis=1)

print(df['module'].value_counts(dropna=False))

In [ ]:
# Columns to keep (from 2.3 analysis)
keep_cols = [
    'unique_key', 'created_date', 'closed_date', 'agency',
    'complaint_type', 'descriptor', 'location_type', 'incident_zip',
    'status', 'borough', 'latitude', 'longitude',
    'open_data_channel_type', 'module'
]

# Filter: only rows that matched one of our 6 modules
df_filtered = df[df['module'].notna()][keep_cols].copy()

print("Filtered shape:", df_filtered.shape)
print("\nModule distribution:")
print(df_filtered['module'].value_counts())

# Save to processed folder
df_filtered.to_csv("../data/processed/nyc311_filtered.csv", index=False)

In [ ]:
print(df_filtered.shape)
df_filtered.head()

In [ ]:
print(df_filtered.shape)

print(df_filtered["module"].value_counts())

df_filtered.head()

In [ ]:
import os

print(os.getcwd())

In [ ]:
print(os.path.exists("data/processed/nyc311_filtered.csv"))
print(os.path.exists("../data/processed/nyc311_filtered.csv"))

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/nyc311_filtered.csv", low_memory=False)
print(df.shape)
df.info()

In [ ]:
import pandas as pd

PROCESSED_DATA = "../data/processed/nyc311_filtered.csv"

df = pd.read_csv(PROCESSED_DATA)

print(df.shape)
print(df["module"].value_counts())
print(df.isnull().mean().sort_values(ascending=False))

In [ ]:
import pandas as pd
raw = pd.read_csv("../data/raw/nyc311_raw.csv", low_memory=False)
print(raw.shape)
raw.head()

In [ ]:
print(df_filtered.shape)
print(df_filtered['module'].value_counts())


In [ ]:
df_filtered.to_csv("../data/processed/nyc311_filtered.csv", index=False)

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/nyc311_filtered.csv", low_memory=False)

print(df.shape)
df.info()

(72472, 14)
<class 'pandas.DataFrame'>
RangeIndex: 72472 entries, 0 to 72471
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   unique_key              72472 non-null  int64  
 1   created_date            72472 non-null  str    
 2   closed_date             70690 non-null  str    
 3   agency                  72472 non-null  str    
 4   complaint_type          72472 non-null  str    
 5   descriptor              72382 non-null  str    
 6   location_type           34808 non-null  str    
 7   incident_zip            71734 non-null  float64
 8   status                  72472 non-null  str    
 9   borough                 72472 non-null  str    
 10  latitude                69502 non-null  float64
 11  longitude               69502 non-null  float64
 12  open_data_channel_type  72472 non-null  str    
 13  module                  72472 non-null  str    
dtypes: float64(3), int64(1), str(10)
memo

In [2]:
print("Total rows:", len(df))
print("Duplicate unique_key rows:", df['unique_key'].duplicated().sum())

df = df.drop_duplicates(subset='unique_key', keep='first')

print("Rows after dedup:", len(df))

Total rows: 72472
Duplicate unique_key rows: 0
Rows after dedup: 72472


In [3]:
null_summary = df.isnull().sum().sort_values(ascending=False)
print(null_summary[null_summary > 0])


location_type    37664
latitude          2970
longitude         2970
closed_date       1782
incident_zip       738
descriptor          90
dtype: int64


In [4]:
# Fill categorical fields
df['descriptor'] = df['descriptor'].fillna('Unknown')
df['location_type'] = df['location_type'].fillna('Unspecified')

# Drop rows missing critical geographic fields
before = len(df)
df = df.dropna(subset=['incident_zip', 'latitude', 'longitude'])
after = len(df)

print(f"Dropped {before - after} rows missing zip/lat/long")
print(f"Remaining rows: {after}")

Dropped 2973 rows missing zip/lat/long
Remaining rows: 69499


In [5]:
df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
df['closed_date'] = pd.to_datetime(df['closed_date'], errors='coerce')

print(df[['created_date', 'closed_date']].dtypes)
print(df[['created_date', 'closed_date']].head())

created_date    datetime64[us]
closed_date     datetime64[us]
dtype: object
         created_date         closed_date
0 2024-12-31 23:56:00 2025-01-02 19:38:00
1 2024-12-31 23:53:00 2025-01-01 09:15:00
2 2024-12-31 23:50:55 2024-12-31 23:56:13
3 2024-12-31 23:50:00 2025-01-02 13:33:00
4 2024-12-31 23:47:27 2025-01-02 12:55:35


In [6]:
# Check the actual range in your data
print(df[['latitude', 'longitude']].describe())

           latitude     longitude
count  69499.000000  69499.000000
mean      40.715665    -73.931242
std        0.082581      0.093815
min       40.499464    -74.254937
25%       40.656205    -73.980598
50%       40.714904    -73.939822
75%       40.767515    -73.873191
max       40.912869    -73.700597


In [7]:
before = len(df)

df = df[
    (df['latitude'].between(40.4, 40.95)) &
    (df['longitude'].between(-74.3, -73.65))
]

after = len(df)
print(f"Dropped {before - after} rows with invalid coordinates")
print(f"Remaining rows: {after}")

Dropped 0 rows with invalid coordinates
Remaining rows: 69499


In [8]:
df = df.rename(columns={
    'incident_zip': 'zip_code',
    'open_data_channel_type': 'submission_channel'
})

print(df.columns.tolist())

['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type', 'descriptor', 'location_type', 'zip_code', 'status', 'borough', 'latitude', 'longitude', 'submission_channel', 'module']


In [9]:
# Check current unique values
print(df['borough'].unique())
print(df['status'].unique())

<StringArray>
['MANHATTAN', 'BROOKLYN', 'STATEN ISLAND', 'QUEENS', 'BRONX', 'Unspecified']
Length: 6, dtype: str
<StringArray>
['Closed', 'Open', 'Started', 'In Progress', 'Assigned', 'Pending']
Length: 6, dtype: str


In [10]:
# Standardize casing and strip whitespace
df['borough'] = df['borough'].str.strip().str.title()
df['status'] = df['status'].str.strip().str.title()
df['location_type'] = df['location_type'].str.strip().str.title()

print(df['borough'].value_counts())

borough
Brooklyn         22248
Queens           19317
Manhattan        13549
Bronx             8729
Staten Island     5650
Unspecified          6
Name: count, dtype: int64


In [11]:
print("Final shape:", df.shape)
print("\nRemaining nulls:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nBorough distribution:")
print(df['borough'].value_counts())

Final shape: (69499, 14)

Remaining nulls:
unique_key               0
created_date             0
closed_date           1664
agency                   0
complaint_type           0
descriptor               0
location_type            0
zip_code                 0
status                   0
borough                  0
latitude                 0
longitude                0
submission_channel       0
module                   0
dtype: int64

Data types:
unique_key                     int64
created_date          datetime64[us]
closed_date           datetime64[us]
agency                           str
complaint_type                   str
descriptor                       str
location_type                    str
zip_code                     float64
status                           str
borough                          str
latitude                     float64
longitude                    float64
submission_channel               str
module                           str
dtype: object

Borough distribution

In [12]:
df.to_csv("../data/processed/nyc311_cleaned.csv", index=False)
print("Saved cleaned dataset:", df.shape)

Saved cleaned dataset: (69499, 14)


In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/processed/nyc311_cleaned.csv", parse_dates=['created_date', 'closed_date'])

print(df.shape)
df.head()

(69499, 14)


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,location_type,zip_code,status,borough,latitude,longitude,submission_channel,module
0,63572271,2024-12-31 23:56:00,2025-01-02 19:38:00,DEP,Noise,Noise: air condition/ventilation equipment (NV1),Unspecified,10023.0,Closed,Manhattan,40.779622,-73.975988,ONLINE,Environment
1,63578403,2024-12-31 23:53:00,2025-01-01 09:15:00,DEP,Sewer,Sewer Backup (Use Comments) (SA),Unspecified,11234.0,Closed,Brooklyn,40.625472,-73.918895,PHONE,Water
2,63582757,2024-12-31 23:50:55,2024-12-31 23:56:13,DPR,Animal in a Park,Dog Off Leash,Park,10314.0,Closed,Staten Island,40.599974,-74.162847,ONLINE,Parks
3,63580613,2024-12-31 23:50:00,2025-01-02 13:33:00,DOT,Street Light Condition,Street Light Out,Unspecified,11417.0,Closed,Queens,40.678583,-73.842153,UNKNOWN,Electricity
4,63578347,2024-12-31 23:47:27,2025-01-02 12:55:35,DSNY,Dirty Condition,Trash,Street,11421.0,Closed,Queens,40.692236,-73.865859,ONLINE,Sanitation


In [2]:
print(df.dtypes[['created_date', 'closed_date']])

created_date    datetime64[us]
closed_date     datetime64[us]
dtype: object


In [3]:
module_counts = df['module'].value_counts().reset_index()
module_counts.columns = ['module', 'count']

fig = px.bar(
    module_counts,
    x='module', y='count',
    title='Complaint Distribution by Module',
    color='module',
    text='count'
)
fig.update_layout(showlegend=False)
fig.show()

In [4]:
borough_counts = df['borough'].value_counts().reset_index()
borough_counts.columns = ['borough', 'count']

fig = px.bar(
    borough_counts,
    x='borough', y='count',
    title='Complaints by Borough',
    color='borough',
    text='count'
)
fig.update_layout(showlegend=False)
fig.show()

In [5]:
df['date_only'] = df['created_date'].dt.date
daily_counts = df.groupby('date_only').size().reset_index(name='count')

fig = px.line(
    daily_counts,
    x='date_only', y='count',
    title='Daily Complaint Volume Over Time'
)
fig.show()

In [6]:
df['year_month'] = df['created_date'].dt.to_period('M').astype(str)
monthly = df.groupby(['year_month', 'module']).size().reset_index(name='count')

fig = px.line(
    monthly,
    x='year_month', y='count', color='module',
    title='Monthly Complaint Trends by Module',
    markers=True
)
fig.show()

In [15]:
sample = df.sample(n=10000, random_state=42)  # random sample for readability

fig = px.scatter_mapbox(
    sample,
    lat='latitude', lon='longitude',
    color='module',
    hover_data=['complaint_type', 'borough'],
    zoom=9,
    height=700,
    title='Complaint Hotspots (Sample of 10,000)'
)
fig.update_layout(mapbox_style="open-street-map")
fig.show()

C:\Users\bhumika nagar\AppData\Local\Temp\ipykernel_8884\2976421285.py:3: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [8]:
# Only use resolved complaints (closed_date not null)
resolved = df.dropna(subset=['closed_date']).copy()
resolved['response_hours'] = (resolved['closed_date'] - resolved['created_date']).dt.total_seconds() / 3600

# Filter out negative/absurd values (data entry errors)
resolved = resolved[(resolved['response_hours'] >= 0) & (resolved['response_hours'] <= 24*90)]  # cap at 90 days

agency_perf = resolved.groupby('agency')['response_hours'].median().sort_values().reset_index()

fig = px.bar(
    agency_perf,
    x='agency', y='response_hours',
    title='Median Response Time by Agency (Hours)'
)
fig.show()

In [9]:
fig = px.histogram(
    resolved,
    x='response_hours',
    nbins=100,
    title='Distribution of Response Times (Hours)'
)
fig.show()

Response time is heavily right-skewed — most complaints resolve within a few days, but a long tail extends to 60-90+ days. This will require a log-transformation of the target variable for response-time prediction models in Phase 6.

In [10]:
df = df[
    (df['latitude'].between(40.49, 40.92)) &
    (df['longitude'].between(-74.26, -73.68))
]
print(df.shape)

(69499, 16)


In [11]:
df.to_csv("../data/processed/nyc311_cleaned.csv", index=False)

In [13]:
print("Before:", df.shape)

df = df[
    (df['latitude'].between(40.49, 40.92)) &
    (df['longitude'].between(-74.26, -73.68))
]

print("After:", df.shape)

# Save immediately
df.to_csv("../data/processed/nyc311_cleaned.csv", index=False)
print("Saved.")

Before: (69499, 16)
After: (69499, 16)
Saved.
